# Bayesian Optimization for Black-Box Functions (Submission 7)

This notebook implements the advanced **"Surgical Calibration"** Bayesian Optimization framework configured for **Round 7 Queries** across 8 independent black-box environments.

### Core Refinements & Structural Scaling for Round 7:
* **High-Fidelity Surrogate Precision:** Employs a robust Matern 5/2 kernel combined with an expansive fitting configuration (**50 restarts inside the GPR engine**) to perfectly tune localized Automatic Relevance Determination (ARD) length-scales.
* **Comprehensive Global Acquisition Maxima Exploration:** The Expected Improvement (EI) surface is searched via **100 randomized multi-starts** of the L-BFGS-B optimizer, mitigating the risks of pinning queries into narrow local traps.
* **Targeted "Surgical Calibration" Jitter Matrix ($\xi$-Mapping):**
  * **$\xi = 0.4000$ (Radical Escapism Jitter):** Set for Channels 1, 3, 4, and 6 to aggressively jar the search space away from non-responsive or deep negative landscape basins.
  * **$\xi = 0.0500$ (Noisy Processing Regularization):** Retained for Channel 2 alongside a higher data regularizer parameter ($\alpha = 10^{-2}$) to separate stochastic signal variance from global trends.
  * **$\xi = 0.0001$ & $\xi = 0.00005$ (Hyper-Exploitation Basins):** Applied to Channels 5 and 7 to execute micro-step refinements within high-yield neighborhoods (such as the 4440.5 coordinate region).
  * **$\xi = 0.0000$ (Pure Maximization Exploitation):** Dedicated strictly to Channel 8 to leverage absolute peak tracking to climb from the current 9.914 peak and break through the 10.0 milestone target boundary.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enable inline visualization tooling
%matplotlib inline 

print("Environment initialized. Round 7 Surgical Calibration runtime active.")

## Hyperparameter Tuning & Bayesian Optimizer Architecture

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Configures regularizer parameters for noise mitigation.
        """
        # alpha=1e-2 for noisy F2 to absorb local variance, 1e-6 for others to trust observations fully
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Extracts historical multi-round arrays and fits the high-precision GPR model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]

        # Matern nu=2.5 provides a robust non-smooth modeling baseline for multi-modal spaces
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        # 50 restarts ensure the length scales (ARD) are perfectly tuned against structural density anomalies
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha, 
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for a targeted exploration jitter (xi).
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Maximizes Expected Improvement across 100 randomized local exploration seeds.
        """
        best_x = None
        max_ei = -1
        # 100 restarts for the acquisition function to reliably extract global EI peaks
        for _ in range(100):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Multi-Target Execution Pipeline & File Exfiltration

Runs through our individual channels sequentially, applying the specific "Surgical Calibration" exploratory map before committing formatted query coordinates to disk.

In [ ]:
output_file = "submission_7_results.txt"

# xi_map Logic: "Surgical Calibration"
# F1, F3, F4, F6: Increased xi to 0.4 to force a jump out of deep negative zones.
# F5: Decreased xi to 0.00005 to hyper-refine the 790/4440 basin.
# F8: xi = 0.0 (Pure Exploitation). We are at 9.914; it's time to hit 10.0.
xi_map = {
    1: 0.4000, 
    2: 0.0500, 
    3: 0.4000, 
    4: 0.4000, 
    5: 0.00005, 
    6: 0.4000, 
    7: 0.0001, 
    8: 0.0000  
}

print("Generating Submission 7: Surgical Calibration...")
print("-" * 55)

notebook_results = {}

for i in range(1, 9):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Path '{func_dir}' not found. Skipping array pipeline.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Save collected coordinates out into deployment format
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 7 compiled successfully into: {output_file}")

### Final Proposed Coordinate Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 7)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")